In [ ]:
!pip install -q "transformers>=4.40,<5.0" accelerate torchaudio librosa scikit-learn

!git clone -q https://github.com/m3hrdadfi/soxan.git
import sys
sys.path.append("/content/soxan")
from src.models import Wav2Vec2ForSpeechClassification

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
GPU available: True
Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoConfig, Wav2Vec2FeatureExtractor

PHASE1_CHECKPOINT = "/content/drive/MyDrive/final_project/shemo_phase1_checkpoint"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = AutoConfig.from_pretrained(PHASE1_CHECKPOINT)
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(PHASE1_CHECKPOINT)
target_sampling_rate = feature_extractor.sampling_rate

model = Wav2Vec2ForSpeechClassification.from_pretrained(PHASE1_CHECKPOINT).to(device)
model.eval()

print("Model labels:", config.id2label)
print("Sampling rate:", target_sampling_rate)

Model labels: {0: 'anger', 1: 'happiness', 2: 'neutral', 3: 'sadness'}
Sampling rate: 16000


In [ ]:
import os

REAL_DATA_DIR = "/content/drive/MyDrive/final_project/voice_dataset"

for root, dirs, files in os.walk(REAL_DATA_DIR):
    level = root.replace(REAL_DATA_DIR, "").count(os.sep)
    if level > 2:
        continue
    indent = "  " * level
    print(f"{indent}{root}/")
    for f in files[:6]:
        print(f"{indent}  {f}")

/content/drive/MyDrive/final_project/voice_dataset/
  metadata.xlsx
  /content/drive/MyDrive/final_project/voice_dataset/audio/
    /content/drive/MyDrive/final_project/voice_dataset/audio/speaker_10/
      022_SAD.m4a
      021_HAP.m4a
      020_ANG.m4a
      037_NEU.m4a
    /content/drive/MyDrive/final_project/voice_dataset/audio/speaker_1/
      025_ANG.m4a
      024_NEU.m4a
      010_SAD.m4a
      023_HAP.m4a
      077_HAP.m4a
    /content/drive/MyDrive/final_project/voice_dataset/audio/speaker_11/
      036_ANG.m4a
      034_SAD.m4a
      032_NEU.m4a
      029_HAP.m4a
    /content/drive/MyDrive/final_project/voice_dataset/audio/speaker_13/
      028_ANG.m4a
      027_NEU.m4a
      026_HAP.m4a
    /content/drive/MyDrive/final_project/voice_dataset/audio/speaker_12/
      035_SAD.m4a
      033_ANG.m4a
      030_NEU.m4a
      031_HAP.m4a
    /content/drive/MyDrive/final_project/voice_dataset/audio/speaker_14/
      011_ANG.m4a
      019_NEU.m4a
      018_SAD.m4a
    /content/drive/My

In [ ]:
# Install ffmpeg for audio conversion (suppress output)
!apt-get -y -q install ffmpeg > /dev/null 2>&1

import glob, os, subprocess

# Find all .m4a files in the real data directory
m4a_files = glob.glob(f"{REAL_DATA_DIR}/**/*.m4a", recursive=True)
print(f"Number of .m4a files found: {len(m4a_files)}")

# Set up output directory for converted WAV files (on Colab temp disk for speed)
CONVERTED_DIR = "/content/real_audio_wav"
os.makedirs(CONVERTED_DIR, exist_ok=True)

# Convert each .m4a file to .wav (16kHz, mono)
for f in m4a_files:
    speaker = os.path.basename(os.path.dirname(f))
    out_dir = os.path.join(CONVERTED_DIR, speaker)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(f))[0] + ".wav")
    if not os.path.exists(out_path):
        subprocess.run(
            ["ffmpeg", "-y", "-i", f, "-ar", "16000", "-ac", "1", out_path],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )

# Count converted files
converted_count = len(glob.glob(f"{CONVERTED_DIR}/**/*.wav", recursive=True))
print(f"Number of converted files: {converted_count}")

Number of .m4a files found: 112
Number of converted files: 112


In [ ]:
from collections import Counter

# Mapping of emotion codes to standardized labels
LABEL_CODE_MAP = {
    "ANG": "anger",
    "HAP": "happiness",
    "NEU": "neutral",
    "SAD": "sadness",
}

def get_label_from_filename(filepath):
    name = os.path.splitext(os.path.basename(filepath))[0].upper()
    for code, emo in LABEL_CODE_MAP.items():
        if code in name:
            return emo
    return None

# Get speaker directories sorted by speaker ID
speaker_dirs = sorted(glob.glob(f"{CONVERTED_DIR}/speaker_*"), key=lambda p: int(p.split("_")[-1]))
print(f"Number of speaker folders: {len(speaker_dirs)}")

real_data = []  # (filepath, label, speaker_id)
for d in speaker_dirs:
    speaker_id = os.path.basename(d)
    for f in glob.glob(f"{d}/*.wav"):
        label = get_label_from_filename(f)
        if label:
            real_data.append((f, label, speaker_id))

print(f"Total labeled samples: {len(real_data)}")
print("Class distribution:", Counter([l for _, l, _ in real_data]))

Number of speaker folders: 26
Total labeled samples: 112
Class distribution: Counter({'happiness': 30, 'sadness': 28, 'anger': 27, 'neutral': 27})


In [ ]:
import torchaudio
import numpy as np

def speech_file_to_array_fn(path, target_sr):
    """Load audio file and resample to target sampling rate"""
    speech_array, orig_sr = torchaudio.load(path)
    if speech_array.shape[0] > 1:
        speech_array = speech_array.mean(dim=0, keepdim=True)  # Convert to mono by averaging channels
    speech = torchaudio.transforms.Resample(orig_sr, target_sr)(speech_array).squeeze().numpy()
    return speech

@torch.no_grad()
def extract_embedding(path):
    """Extract Wav2Vec2 embedding from audio file"""
    speech = speech_file_to_array_fn(path, target_sampling_rate)
    inputs = feature_extractor(speech, sampling_rate=target_sampling_rate, return_tensors="pt", padding=True)
    input_values = inputs["input_values"].to(device)
    hidden_states = model.wav2vec2(input_values).last_hidden_state
    pooled = hidden_states.mean(dim=1).squeeze(0).cpu().numpy()  # Global average pooling
    return pooled

# Extract embeddings for all real data samples
embeddings, labels, speakers = [], [], []
for i, (path, label, speaker_id) in enumerate(real_data):
    embeddings.append(extract_embedding(path))
    labels.append(label)
    speakers.append(speaker_id)
    if (i + 1) % 20 == 0:
        print(f"{i+1}/{len(real_data)} processed")

embeddings = np.stack(embeddings)
print("Embedding matrix shape:", embeddings.shape)

20/112 processed
40/112 processed
60/112 processed
80/112 processed
100/112 processed
Embedding matrix shape: (112, 1024)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Get unique speakers sorted by ID
unique_speakers = sorted(set(speakers), key=lambda s: int(s.split("_")[-1]))
print(f"Number of speakers: {len(unique_speakers)}")

# Leave-one-speaker-out cross-validation
all_true, all_pred, all_speaker_log = [], [], []

for test_speaker in unique_speakers:
    # Split data: train on all speakers except the test speaker
    train_idx = [i for i, s in enumerate(speakers) if s != test_speaker]
    test_idx = [i for i, s in enumerate(speakers) if s == test_speaker]
    if not test_idx:
        continue

    X_train, y_train = embeddings[train_idx], [labels[i] for i in train_idx]
    X_test, y_test = embeddings[test_idx], [labels[i] for i in test_idx]

    # Standardize features
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    # Train logistic regression classifier
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
    clf.fit(X_train_s, y_train)
    preds = clf.predict(X_test_s)

    # Store results
    all_true.extend(y_test)
    all_pred.extend(preds)
    all_speaker_log.extend([test_speaker] * len(y_test))

print(f"Total evaluated samples: {len(all_true)}")

Number of speakers: 26
Total evaluated samples: 112


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

print(f"LOSO accuracy on real data: {accuracy_score(all_true, all_pred):.4f}")
print(classification_report(all_true, all_pred))

# Display confusion matrix
labels_order = sorted(set(all_true) | set(all_pred))
cm = confusion_matrix(all_true, all_pred, labels=labels_order)
pd.DataFrame(cm, index=labels_order, columns=labels_order)

LOSO accuracy on real data: 0.6696
              precision    recall  f1-score   support

       anger       0.62      0.56      0.59        27
   happiness       0.79      0.63      0.70        30
     neutral       0.61      0.74      0.67        27
     sadness       0.68      0.75      0.71        28

    accuracy                           0.67       112
   macro avg       0.68      0.67      0.67       112
weighted avg       0.68      0.67      0.67       112



,anger,happiness,neutral,sadness
anger,15,3,5,4
happiness,4,19,5,2
neutral,1,2,20,4
sadness,4,0,3,21


In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Note: Using the simple embeddings (last layer from cell 6), not embeddings_v2
# Train final classifier on all data
final_scaler = StandardScaler()
X_all_s = final_scaler.fit_transform(embeddings)

final_clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
final_clf.fit(X_all_s, labels)

# Save the final model and scaler
FINAL_DIR = "/content/drive/MyDrive/final_project/final_classifier"
os.makedirs(FINAL_DIR, exist_ok=True)

joblib.dump(final_clf, f"{FINAL_DIR}/logistic_classifier.joblib")
joblib.dump(final_scaler, f"{FINAL_DIR}/scaler.joblib")

print(f"Final model saved to: {FINAL_DIR}")
print("Classes:", final_clf.classes_)

Final model saved to: /content/drive/MyDrive/final_project/final_classifier
Classes: ['anger' 'happiness' 'neutral' 'sadness']
